# Beginner 02 — Humans, Workloads and Agents

## Enterprise scenario

An employee uses a travel agent. The logical agent runs in several environments and invokes internal tools plus a booking sub-agent.

We will build an explicit identity model and demonstrate what breaks when identities are collapsed.

### Lab outcomes
- model human, application, agent, workload, service, and resource identities;
- model ownership separately from execution;
- map approved workloads to logical agents;
- propagate requester/actor/workload context;
- demonstrate environment isolation;
- expose shared-account attribution failures;
- model a multi-agent actor chain;
- implement registry lifecycle controls.


In [ ]:
from dataclasses import dataclass, field, replace
from datetime import datetime, timezone
from enum import Enum
from typing import Optional
import json, uuid

def now():
    return datetime.now(timezone.utc)


## 1 — Identity taxonomy

In [ ]:
class IdentityType(str, Enum):
    HUMAN = "human"
    APPLICATION = "application"
    AGENT = "agent"
    WORKLOAD = "workload"
    SERVICE = "service"
    RESOURCE = "resource"

@dataclass
class Identity:
    id: str
    type: IdentityType
    name: str
    owner: Optional[str] = None
    environment: Optional[str] = None
    status: str = "active"
    metadata: dict = field(default_factory=dict)

alice = Identity("user:alice", IdentityType.HUMAN, "Alice")
portal = Identity("client:travel-portal", IdentityType.APPLICATION, "Travel Portal", owner="team:travel")
agent = Identity("agent:travel-planner", IdentityType.AGENT, "Travel Planner", owner="team:travel")
prod = Identity(
    "spiffe://corp.example/prod/travel-agent",
    IdentityType.WORKLOAD, "Travel Agent Production Workload",
    owner="team:travel", environment="prod"
)
policy_api = Identity("service:travel-policy", IdentityType.SERVICE, "Travel Policy API", owner="team:travel")
trip = Identity("trip:483", IdentityType.RESOURCE, "Alice Conference Trip", owner=alice.id)

for item in [alice, portal, agent, prod, policy_api, trip]:
    print(f"{item.type.value:12} {item.id}")


## 2 — Relationships are not identity equality

`Alice owns/uses/delegates to Travel Planner` does not mean `Travel Planner == Alice`.

Represent relationships explicitly.


In [ ]:
@dataclass(frozen=True)
class Relationship:
    source: str
    relation: str
    target: str

relationships = [
    Relationship("team:travel", "owns", agent.id),
    Relationship(alice.id, "delegates_to", agent.id),
    Relationship(agent.id, "runs_as", prod.id),
    Relationship(agent.id, "invokes", policy_api.id),
    Relationship(alice.id, "owns", trip.id),
]

for r in relationships:
    print(r.source, f"--{r.relation}-->", r.target)


## 3 — Agent registry

The registry is a governance view, not an identity provider.

In [ ]:
class AgentRegistry:
    def __init__(self):
        self.identities = {}
        self.approved_workloads = {}

    def register(self, identity):
        if identity.id in self.identities:
            raise ValueError("identity already registered")
        self.identities[identity.id] = identity

    def approve_workload(self, agent_id, workload_id):
        a = self.identities[agent_id]
        w = self.identities[workload_id]
        if a.type != IdentityType.AGENT or w.type != IdentityType.WORKLOAD:
            raise TypeError("mapping must be agent -> workload")
        self.approved_workloads.setdefault(agent_id, set()).add(workload_id)

    def workload_is_approved(self, agent_id, workload_id):
        return workload_id in self.approved_workloads.get(agent_id, set())

    def disable(self, identity_id):
        self.identities[identity_id].status = "disabled"

registry = AgentRegistry()
for item in [alice, portal, agent, prod, policy_api, trip]:
    registry.register(item)
registry.approve_workload(agent.id, prod.id)

print(registry.workload_is_approved(agent.id, prod.id))


## 4 — Environment-specific workload identities

In [ ]:
dev = Identity(
    "spiffe://corp.example/dev/travel-agent",
    IdentityType.WORKLOAD,
    "Travel Agent Development Workload",
    owner="team:travel",
    environment="dev"
)
registry.register(dev)

print("prod approved:", registry.workload_is_approved(agent.id, prod.id))
print("dev approved :", registry.workload_is_approved(agent.id, dev.id))


The logical agent name is the same, but the runtime is not. A production resource can require an approved production workload rather than trusting `agent:travel-planner` alone.

## 5 — Explicit call context

In [ ]:
@dataclass(frozen=True)
class CallContext:
    requester: str
    actor: str
    workload: str
    task_id: str
    parent_actor: Optional[str] = None
    actor_chain: tuple[str, ...] = ()

ctx = CallContext(
    requester=alice.id,
    actor=agent.id,
    workload=prod.id,
    task_id="trip:483",
    actor_chain=(alice.id, agent.id),
)
ctx


## 6 — Immediate caller versus business actor

In [ ]:
@dataclass(frozen=True)
class ServiceRequest:
    authenticated_peer: str
    requester: str
    actor: str
    task: str

request = ServiceRequest(
    authenticated_peer=prod.id,
    requester=alice.id,
    actor=agent.id,
    task=ctx.task_id,
)

print("Who directly authenticated? ", request.authenticated_peer)
print("Who requested the work?     ", request.requester)
print("Which logical agent acts?   ", request.actor)


All three can be true simultaneously. Mature authorization can use the authenticated immediate caller plus trusted delegated context rather than pretending they are one identity.

## 7 — Validate logical-agent/workload binding

In [ ]:
def validate_runtime(registry, ctx):
    logical = registry.identities.get(ctx.actor)
    workload = registry.identities.get(ctx.workload)
    if not logical or logical.status != "active":
        return False, "logical agent inactive or unknown"
    if not workload or workload.status != "active":
        return False, "workload inactive or unknown"
    if not registry.workload_is_approved(ctx.actor, ctx.workload):
        return False, "workload is not approved for logical agent"
    return True, "approved agent/workload binding"

print(validate_runtime(registry, ctx))

dev_ctx = replace(ctx, workload=dev.id)
print(validate_runtime(registry, dev_ctx))


## 8 — Why one shared service account is insufficient

In [ ]:
events_bad = [
    {"principal": "service-account:agents", "action": "book", "trip": "483"},
    {"principal": "service-account:agents", "action": "cancel", "trip": "902"},
]
print(json.dumps(events_bad, indent=2))


From these records, can you determine which logical agent acted, which user initiated the intent, or which runtime executed it? No.

A shared infrastructure identity may still exist, but it cannot be your only attribution mechanism.


## 9 — Better audit context

In [ ]:
def audit_event(ctx, action, resource, authenticated_peer=None):
    return {
        "timestamp": now().isoformat(),
        "requester": ctx.requester,
        "actor": ctx.actor,
        "workload": ctx.workload,
        "authenticated_peer": authenticated_peer or ctx.workload,
        "task": ctx.task_id,
        "parent_actor": ctx.parent_actor,
        "actor_chain": list(ctx.actor_chain),
        "action": action,
        "resource": resource,
    }

print(json.dumps(audit_event(ctx, "flight:search", "trip:483"), indent=2))


## 10 — Multi-agent chain

In [ ]:
booking_agent = Identity(
    "agent:booking-specialist", IdentityType.AGENT,
    "Booking Specialist", owner="team:travel"
)
booking_workload = Identity(
    "spiffe://corp.example/prod/booking-agent",
    IdentityType.WORKLOAD, "Booking Agent Production Workload",
    owner="team:travel", environment="prod"
)
registry.register(booking_agent)
registry.register(booking_workload)
registry.approve_workload(booking_agent.id, booking_workload.id)

child_ctx = CallContext(
    requester=ctx.requester,
    actor=booking_agent.id,
    workload=booking_workload.id,
    task_id=ctx.task_id,
    parent_actor=ctx.actor,
    actor_chain=ctx.actor_chain + (booking_agent.id,),
)

print(json.dumps(audit_event(child_ctx, "flight:reserve", "trip:483"), indent=2))


Notice that the child did not become Alice or the supervisor agent. Its own actor identity is preserved while the chain records why it is involved.

## 11 — Lifecycle: disable one workload without disabling the agent

In [ ]:
print("before:", validate_runtime(registry, ctx))
registry.disable(prod.id)
print("after :", validate_runtime(registry, ctx))
print("logical agent status:", registry.identities[agent.id].status)


This illustrates separate lifecycles. A compromised deployment can be disabled while the logical agent remains registered and can later be mapped to a clean workload.

## 12 — Challenge: build a stricter production admission check

In [ ]:
def production_admission_check(registry, ctx):
    # TODO:
    # 1. requester must be active
    # 2. actor must be an active AGENT
    # 3. workload must be active and type WORKLOAD
    # 4. workload environment must equal "prod"
    # 5. workload must be approved for actor
    # Return (bool, reason)
    raise NotImplementedError


## 13 — Challenge: identity collision analysis

Suppose four agents all run behind:

```text
client_id = enterprise-agent-platform
service_account = agents-prod
```

Answer:

1. Which identity is useful for authenticating the platform?
2. Which information is missing for agent-level authorization?
3. How would you preserve independent agent revocation?
4. How would you prove which workload executed a suspicious call?
5. What should appear in the audit event?

## 14 — Review questions

1. Why is an OAuth client not automatically the same thing as a logical agent?
2. Why is a service account not automatically the same thing as a workload instance?
3. What does workload attestation establish?
4. What is the difference between a SPIFFE ID and an SVID?
5. Why should development and production use distinguishable identities?
6. Why is ownership a relationship instead of identity equality?
7. In a multi-agent chain, which actor should a downstream service see?
8. What is gained by keeping both immediate caller and business requester?

## Next course

**Beginner 03 — Authentication, Credentials and Tokens**
